# Painel Interativo de Análise de Vendas
## Etapa 1 — Preparação e Importação de Dados

In [ ]:
# 1) Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('graficos', exist_ok=True)

In [ ]:
# 2) Criação do dataset fictício vendas.csv
np.random.seed(42)
n = 500

lojas    = ['Loja SP', 'Loja RJ', 'Loja MG', 'Loja RS']
regioes  = {'Loja SP': 'Sudeste', 'Loja RJ': 'Sudeste', 'Loja MG': 'Sudeste', 'Loja RS': 'Sul'}
produtos = ['Notebook', 'Smartphone', 'Tablet', 'Monitor', 'Teclado', 'Mouse', 'Headset']
categorias = {'Notebook': 'Computadores', 'Smartphone': 'Celulares', 'Tablet': 'Celulares',
              'Monitor': 'Computadores', 'Teclado': 'Periféricos', 'Mouse': 'Periféricos', 'Headset': 'Periféricos'}

loja_col     = np.random.choice(lojas, n)
produto_col  = np.random.choice(produtos, n)
quantidade   = np.random.randint(1, 20, n).astype(float)
preco        = np.round(np.random.uniform(50, 5000, n), 2)
datas        = pd.date_range('2023-01-01', '2023-12-31', periods=n)

# Introduz ~3% de nulos para simular dados reais
idx_null_q = np.random.choice(n, 15, replace=False)
idx_null_p = np.random.choice(n, 15, replace=False)
quantidade[idx_null_q] = np.nan
preco[idx_null_p]      = np.nan

df = pd.DataFrame({
    'data':           datas,
    'loja':           loja_col,
    'regiao':         [regioes[l] for l in loja_col],
    'produto':        produto_col,
    'categoria':      [categorias[p] for p in produto_col],
    'quantidade':     quantidade,
    'preco_unitario': preco
})

df['data'] = pd.to_datetime(df['data'])   # garante tipo datetime
df.to_csv('vendas.csv', index=False)
print('Dataset criado:', df.shape)
df.head()

In [ ]:
# 3) Nova coluna faturamento
df['faturamento'] = df['quantidade'] * df['preco_unitario']
df[['quantidade', 'preco_unitario', 'faturamento']].head()

In [ ]:
# 4) Verificação de integridade
print('=== .info() ===')
df.info()
print('\n=== Valores nulos ===')
print(df.isnull().sum())
print('\n=== .describe() ===')
df.describe()

In [ ]:
# 5) Tratamento de valores nulos pela média da coluna
df['quantidade']     = df['quantidade'].fillna(df['quantidade'].mean())
df['preco_unitario'] = df['preco_unitario'].fillna(df['preco_unitario'].mean())
df['faturamento']    = df['quantidade'] * df['preco_unitario']   # recalcula após imputação

print('Nulos restantes:')
print(df.isnull().sum())

## Etapa 2 — Análise Exploratória e Estatísticas

In [ ]:
# 6) Faturamento total por loja (maior → menor)
fat_loja = df.groupby('loja')['faturamento'].sum().sort_values(ascending=False)
print('Faturamento por loja:')
print(fat_loja.apply(lambda x: f'R$ {x:,.2f}'))

In [ ]:
# 7) Ticket médio por loja
ticket_medio = df.groupby('loja').apply(
    lambda g: g['faturamento'].sum() / len(g)
).sort_values(ascending=False)
print('Ticket médio por loja:')
print(ticket_medio.apply(lambda x: f'R$ {x:,.2f}'))

In [ ]:
# 8) Top 5 produtos mais vendidos em quantidade
top5 = df.groupby('produto')['quantidade'].sum().sort_values(ascending=False).head(5)
print('Top 5 produtos por quantidade:')
print(top5)

In [ ]:
# 9) Mês com maior e menor faturamento
fat_mensal = df.groupby(df['data'].dt.to_period('M'))['faturamento'].sum()
mes_max = fat_mensal.idxmax().strftime('%m/%Y')
mes_min = fat_mensal.idxmin().strftime('%m/%Y')
print(f'Maior faturamento: {mes_max} — R$ {fat_mensal.max():,.2f}')
print(f'Menor faturamento: {mes_min} — R$ {fat_mensal.min():,.2f}')

In [ ]:
# 10) Coluna mês
df['mes'] = df['data'].dt.month
df[['data', 'mes']].head()

## Etapa 3 — Visualizações com matplotlib e seaborn

In [ ]:
# 11) Gráfico de barras — Faturamento por loja
fig, ax = plt.subplots(figsize=(8, 5))
cores = sns.color_palette('deep', len(fat_loja))
ax.bar(fat_loja.index, fat_loja.values, color=cores)
ax.set_title('Faturamento Total por Loja', fontsize=14, fontweight='bold')
ax.set_xlabel('Loja')
ax.set_ylabel('Faturamento (R$)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
for bar, val in zip(ax.patches, fat_loja.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'R$ {val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('faturamento_por_loja.png', dpi=150)
plt.savefig('graficos/faturamento_por_loja.png', dpi=150)
plt.show()

In [ ]:
# 12) Gráfico de linhas — Evolução mensal do faturamento
fat_mensal_plot = df.groupby('mes')['faturamento'].sum()
meses_label = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fat_mensal_plot.index, fat_mensal_plot.values, marker='o', color='steelblue', linewidth=2)
for x, y in zip(fat_mensal_plot.index, fat_mensal_plot.values):
    ax.annotate(f'R$ {y:,.0f}', (x, y), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=7)
ax.set_title('Evolução Mensal do Faturamento (2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Faturamento (R$)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_label)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.tight_layout()
plt.savefig('graficos/evolucao_mensal.png', dpi=150)
plt.show()

In [ ]:
# 13) Gráfico de pizza — Distribuição de faturamento por região
fat_regiao = df.groupby('regiao')['faturamento'].sum()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(fat_regiao.values, labels=fat_regiao.index, autopct='%1.1f%%',
       colors=sns.color_palette('deep', len(fat_regiao)), startangle=140,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Distribuição de Faturamento por Região', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/faturamento_por_regiao.png', dpi=150)
plt.show()

In [ ]:
# 14) Scatter — Relação entre quantidade e faturamento
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x='quantidade', y='faturamento', hue='loja',
                palette='deep', alpha=0.7, ax=ax)
ax.set_title('Relação entre Quantidade Vendida e Faturamento', fontsize=13, fontweight='bold')
ax.set_xlabel('Quantidade')
ax.set_ylabel('Faturamento (R$)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.tight_layout()
plt.savefig('graficos/scatter_qtd_faturamento.png', dpi=150)
plt.show()

In [ ]:
# 15) Boxplot — Distribuição de preços por categoria
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='categoria', y='preco_unitario',
            palette='deep', ax=ax)
ax.set_title('Distribuição de Preços Unitários por Categoria', fontsize=13, fontweight='bold')
ax.set_xlabel('Categoria')
ax.set_ylabel('Preço Unitário (R$)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.tight_layout()
plt.savefig('graficos/boxplot_preco_categoria.png', dpi=150)
plt.show()

In [ ]:
# 16) Heatmap — Correlação entre variáveis numéricas
corr = df[['quantidade', 'preco_unitario', 'faturamento', 'mes']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlação entre Variáveis Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/heatmap_correlacao.png', dpi=150)
plt.show()

## Etapa 4 — Estilização e Apresentação

In [ ]:
# 17) Tema global
sns.set_style('whitegrid')
sns.set_palette('deep')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Tema global aplicado: whitegrid + paleta deep')

In [ ]:
# 18) Revisão de rótulos — regráfico de barras com tema aplicado
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=fat_loja.index, y=fat_loja.values, palette='deep', ax=ax)
ax.set_title('Faturamento Total por Loja (2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Loja', fontsize=12)
ax.set_ylabel('Faturamento (R$)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
for p in ax.patches:
    ax.annotate(f'R$ {p.get_height():,.0f}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('graficos/faturamento_loja_estilizado.png', dpi=150)
plt.show()

In [ ]:
# 19) Layout com múltiplos gráficos — Faturamento por loja e por região
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparativo de Faturamento — Loja vs Região (2023)', fontsize=14, fontweight='bold')

# Painel esquerdo: por loja
sns.barplot(x=fat_loja.index, y=fat_loja.values, palette='deep', ax=axes[0])
axes[0].set_title('Por Loja')
axes[0].set_xlabel('Loja')
axes[0].set_ylabel('Faturamento (R$)')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
for p in axes[0].patches:
    axes[0].annotate(f'R$ {p.get_height():,.0f}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=8)

# Painel direito: por região
axes[1].pie(fat_regiao.values, labels=fat_regiao.index, autopct='%1.1f%%',
            colors=sns.color_palette('deep', len(fat_regiao)),
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Por Região')

plt.tight_layout()
plt.savefig('graficos/comparativo_loja_regiao.png', dpi=150)
plt.show()

In [ ]:
# 20) Exportação e relatório final

# Resumo analítico consolidado
resumo = pd.DataFrame({
    'loja':           fat_loja.index,
    'faturamento_total': fat_loja.values,
    'ticket_medio':   ticket_medio.reindex(fat_loja.index).values
})

resumo_mensal = fat_mensal_plot.reset_index()
resumo_mensal.columns = ['mes', 'faturamento_mensal']

resumo_regiao = fat_regiao.reset_index()
resumo_regiao.columns = ['regiao', 'faturamento_regiao']

resumo.to_csv('resumo_analitico.csv', index=False)
resumo_mensal.to_csv('graficos/resumo_mensal.csv', index=False)
resumo_regiao.to_csv('graficos/resumo_regiao.csv', index=False)

print('=== Relatório Final ===')
print(resumo.to_string(index=False))
print(f'\nMês de maior faturamento : {mes_max}')
print(f'Mês de menor faturamento : {mes_min}')
print(f'\nTop 5 produtos mais vendidos:')
print(top5.to_string())
print('\nArquivos salvos em graficos/ e resumo_analitico.csv')

# Observações analíticas
print("""
=== Observações ===
- O gráfico de barras revela qual loja lidera em faturamento absoluto.
- A evolução mensal indica sazonalidade, com picos em determinados meses.
- O gráfico de pizza mostra a concentração regional do faturamento.
- O scatter evidencia correlação positiva entre quantidade e faturamento,
  com outliers de alto preço unitário gerando faturamentos elevados mesmo
  com baixa quantidade.
- O boxplot expõe a amplitude de preços por categoria, destacando
  Computadores como a categoria de maior variação.
- O heatmap confirma alta correlação entre quantidade e faturamento,
  enquanto preço_unitario apresenta correlação moderada.
""")